In [ ]:
# ustawienie środowiska dla MITSUBA 3 i PyTorch na GPU, jeśli jest dostępne
import mitsuba as mi
try:
    mi.set_variant('cuda_ad_mono_polarized')
    print(f"Wymuszono wariant GPU: {mi.variant()}")
except Exception as e:
    print(f"Nie udało się wymusić GPU: {e}")

import drjit as dr
import torch

if torch.cuda.is_available():
    torch.set_default_device('cuda')
    print("Domyślne urządzenie PyTorch ustawione")

In [ ]:
# Importowanie najwazniejszych komponentow Sionna RT
from sionna.rt import PathSolver
from sionna_env import setup_scene, update_object_position
import ofdm_config as cfg
from IPython.display import clear_output
import numpy as np

scene_file = "./OpenStreetMap/kampus_AGH_test_all/Kampus_AGH_B9_test_all.xml"

scene, tx, rx = setup_scene(scene_file, cfg.fc, cfg.rx_rows, cfg.rx_cols)

dron_target = scene.get("dron_target")
# print(f'Pozycja drona: {dron_target.position}')

samochod_target_b = scene.get("samochod_target_b")
# print(f'Pozycja samochodu_b: {samochod_target_b.position}')

samochod_target_d = scene.get("samochod_target_d")
# print(f'Pozycja samochodu_d: {samochod_target_d.position}')

czlowiek_target = scene.get("czlowiek_target")
# print(f'Pozycja czlowieka: {czlowiek_target.position}')

print(f'Pomyślnie załadowano scenę: {scene_file}')

In [ ]:
# Sprawdzenie materiałów radiowych obiektów w scenie
for name, obj in scene.objects.items():
    if "samochod" in name:
        print(name)
        print(obj.radio_material)
    elif "dron" in name:
        print(name)
        print(obj.radio_material)
    elif "czlowiek" in name:
        print(name)
        print(obj.radio_material)

In [ ]:
p_solver = PathSolver()

num_steps = 200 # Liczba kroków symulacji
frame_time = 0.1  # Czas trwania jednej klatki w sekundach

historia_taps = []

# pętla symulacyjna
for i in range(num_steps):
    clear_output(wait=True)
    print(f"Obliczanie kroku {i+1}/{num_steps}")

    # Ruch samochodu D
    if i <= 190:
        if i > 180:
            samochod_target_d_vel = np.array([-6.4, 0, 0], dtype=float)
        elif i > 170:
            samochod_target_d_vel = np.array([-5, 0, 0], dtype=float)
        elif i > 80:
            samochod_target_d_vel = np.array([0, 8.8, 0], dtype=float)
        elif i > 50:
            samochod_target_d_vel = np.array([14, 0, 0], dtype=float)
        else:
            samochod_target_d_vel = np.array([16, 0, 0], dtype=float)

        update_object_position(samochod_target_d, samochod_target_d_vel, frame_time)
    print(f'Pozycja samochodu_d: {samochod_target_d.position}')

    # Obliczanie ścieżek propagacji
    paths = p_solver(scene=scene,
                    max_depth=4,
                    samples_per_src=5_000_000,
                    los=True,
                    specular_reflection=True,
                    diffuse_reflection=True,
                    edge_diffraction=True,
                    refraction=False,
                    synthetic_array=False,
                    seed=67)

    # Wymuszenie obliczeń na GPU
    dr.eval(paths.a, paths.tau)

    # Zbieranie danych podział na chunki
    chunk_size = 8  
    total_symbols = cfg.num_symbols
    a_chunks = []
    tau_combined = None

    print(f"  -> Przetwarzanie taps w pakietach po {chunk_size} symboli...")
    for start_sym in range(0, total_symbols, chunk_size):
        end_sym = min(start_sym + chunk_size, total_symbols)
        current_chunk_symbols = end_sym - start_sym
        
        # Generowanie fragmentu taps dla osi czasu
        a_chunk, tau_chunk = paths.taps(bandwidth=cfg.bandwidth,
                     num_time_steps=current_chunk_symbols,
                     sampling_frequency=cfg.ofdm_symbol_rate, 
                     l_min = -6,
                     l_max = cfg.l_max_dynamic)
        
        # Wymuszenie ewaluacji tego konkretnego chunku na GPU
        dr.eval(a_chunk, tau_chunk)
        
        # Zrzucenie obliczonego kawałka z GPU do pamięci RAM jako tablica NumPy
        a_chunks.append(a_chunk.numpy())
        
        if tau_combined is None:
            tau_combined = tau_chunk.numpy()

    # Połączenie wszystkich kawałków w jedną tablicę taps
    a_combined = np.concatenate(a_chunks, axis=4)

    historia_taps.append((a_combined, tau_combined))
    print(f"  ✓ Sukces! Zakończono krok {i+1}. Wygenerowano macierz amplitud o kształcie: {a_combined.shape}")

    # 
    # a_final = historia_taps[0][0]   # Macierz zespolona amplitud
    # tau_final = historia_taps[0][1] # Macierz opóźnień

        # # Renderowanie w trakcie pętli
    # if i == 199:

    #     # Opcja renderowania z promieniami w scenie  
    #     # fig = scene.preview(paths=paths, 
    #     #       resolution=(800, 600),
    #     #       rm_vmin=-150,
    #     #       rm_vmax=-6)

    #     # Renderowanie bez promieni w scenie
    #     fig = scene.preview(resolution=(800, 600))

    #     user_input = input("Wciśnij [Enter], aby wykonać następny krok (lub wpisz 'stop', aby przerwać): ")
    #     if user_input.strip().lower() == 'stop':
    #             print("Przerwarzasz symulację ręcznie.")
    #             break

# Zapis danych
np.savez_compressed("historia_taps_car_D.npz",
                     dane=np.array(historia_taps, dtype=object))

print(f"Zapisano dane do pliku 'historia_taps_car_D.npz'.")

In [ ]:
# Renderowanie Mapy osobno dla podglądu
scene.preview(paths=paths, 
              resolution=(800, 600),
              rm_vmin=-150,
              rm_vmax=-6)